In [1]:
import os
import sys
# Agregar el directorio raíz del proyecto al path
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..','..'))
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)
    
from services.DataFrame_maker import *
from services.InteractivePlotter import InteractivePlotterTk
from services.carga_de_datos import *   
from services.utilidades import *
from services.comprobar_netcdf import *

In [2]:
# Recargar módulos locales (ejecutar esta celda cuando cambies código en services/)
import importlib
import sys

# Lista de módulos locales a recargar
modulos_locales = [
    'services.carga_de_datos',
    'services.DataFrame_maker',
    'services.prueba',
    'services.utilidades',
    'services.get_tspan',
    'services.polar2uv',
    'services.uv2polar'
]

for modulo in modulos_locales:
    if modulo in sys.modules:
        importlib.reload(sys.modules[modulo])
        print(f"✓ Recargado: {modulo}")
    else:
        print(f"- No cargado: {modulo}")

# Re-importar después de recargar
from services.DataFrame_maker import *
from services.InteractivePlotter import InteractivePlotterTk
from services.carga_de_datos import *

print("\n¡Módulos locales recargados!")

✓ Recargado: services.carga_de_datos
✓ Recargado: services.DataFrame_maker
- No cargado: services.prueba
✓ Recargado: services.utilidades
✓ Recargado: services.get_tspan
✓ Recargado: services.polar2uv
✓ Recargado: services.uv2polar

¡Módulos locales recargados!


In [3]:
# %matplotlib tk

In [4]:
# Iniciar un diccionario vacío para almacenar los DataFrames de cada boya
dict_de_boyas = {}

In [5]:
# listar_funciones_de_carga()

### 1. Meteo

In [6]:
# 1.3. p1: Crudo unido .pkl
nombre = "BMT2-01-T20_pkl"
nombre_de_archivo = "msj1_crudo_unido_realt_2026022817_2026071023.pkl"
ruta_a_carpeta = "\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_3.2\\2026\\07. Julio\\BMT2-01-T20\\DATOS"
df_pkl_C = cargar_pickle_meteo(ruta_a_carpeta, nombre_de_archivo)
dict_de_boyas[nombre] = df_pkl_C # Guardo el dataframe dentro del diccionario de boyas.
df_pkl_C.head(2)

,tspan,Pa,Ta,Humedad relativa (sonico),Punto de Rocio,Rap2,Dir2,u (mecanico),v (mecanico),R5s2,...,u (sonico),v (sonico),R5s1,R1s1,Temperatura (sonico),Presion (sonico),HR,Lluvia acumulada,Duracion de la lluvia,Intensidad de la lluvia
0,2026-02-28 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-02-28 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# 1.4. Cargar NETCDF 
nombre = "BMT2-01-T20_NC"
nombre_de_archivo = "BMT2-01-T20-METEO-NS506930-A4.0-REALT-20260228-20260710.nc"
df_nc, nc = cargar_nc_meteo(ruta_a_carpeta, nombre_de_archivo)
dict_de_boyas[nombre] = df_nc # Guardo el dataframe dentro del diccionario de boyas.
df_nc.head(5)

,tspan,Pa,Ta,HR,Rap1,Dir1,R5s1,R1s1,R5s2,R1s2,Rap2,Dir2
0,2026-02-28 20:00:00,1011.35,25.2,88.0,2.5,314.0,3.6,3.9,3.3,3.43,2.5,303.0
1,2026-02-28 21:00:00,1010.77,25.3,86.9,4.0,330.2,4.8,5.4,4.6,4.87,3.8,316.0
2,2026-02-28 22:00:00,1010.45,25.1,85.7,5.8,324.8,6.8,7.2,6.7,6.86,5.6,314.0
3,2026-02-28 23:00:00,1010.49,25.1,86.8,6.9,333.1,7.8,8.3,7.5,8.01,6.7,331.0
4,2026-03-01 00:00:00,1010.89,24.5,87.5,6.7,330.8,8.0,8.4,7.5,7.92,6.5,329.0


In [8]:
# Otras revisiones del NC:
# comprobar_nombre_nc
comprobar_nombre_nc(nombre_de_archivo, nc)
comprobar_porcentaje_de_variable(nc, ruta_a_carpeta, "BMT2-01-T20_datos_word_2026022820_2026071023.xlsx", variables="meteo")

NAM correcto


,Variable,esperados,validados,porcentaje_validados
0,Pa,3172,3149,99.274905
1,Ta,3172,3147,99.211854
2,HR,3172,3149,99.274905
3,Rap1,3172,3149,99.274905
4,Dir1,3172,3149,99.274905
5,R1s1,3172,3149,99.274905
6,R5s1,3172,3149,99.274905
7,Rap2,3172,3149,99.274905
8,Dir2,3172,3149,99.274905
9,R1s2,3172,3149,99.274905


In [9]:
# 1.5. p2: Cargar telemetría de BMT3-03-T80 
nombre = "BMT3-03-T80"
df_pkl = pd.DataFrame()  # Inicializar un DataFrame vacío para almacenar los datos combinados
for mes in ["01. Enero","02. Febrero","03. Marzo","04. Abril","05. Mayo", "06. Junio"]:
    ruta_a_carpeta = f"\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_4.4\\2026\\{mes}\\BMT3-03-T80\\DATOS"
    archivos = os.listdir(ruta_a_carpeta)
    for archivo in archivos:
        if archivo.endswith(".pkl"):
            if archivo.startswith("msj1_crudo_unido_realt"):
                df_pkl_crudo_realt = cargar_pickle_meteo(ruta_a_carpeta, archivo)
                df_pkl = pd.concat([df_pkl, df_pkl_crudo_realt], ignore_index=True)
    
df_pkl = df_pkl.reset_index(drop=True)  # Reiniciar el índice del DataFrame combinado
dict_de_boyas[nombre] = df_pkl # Guardo el dataframe combinado dentro del diccionario de boyas.

In [10]:

# 1.5. p2: Cargar telemetría de BMT3-03-T80 
nombre = "BMT3-04-T45"
df_pkl = pd.DataFrame()  # Inicializar un DataFrame vacío para almacenar los datos combinados
for mes in ["01. Enero","02. Febrero","03. Marzo","04. Abril","05. Mayo", "06. Junio"]:
    ruta_a_carpeta = f"\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_4.4\\2026\\{mes}\\BMT3-04-T45\\DATOS"
    archivos = os.listdir(ruta_a_carpeta)
    for archivo in archivos:
        if archivo.endswith(".pkl"):
            if archivo.startswith("msj1_crudo_unido_realt"):
                df_pkl_crudo_realt = cargar_pickle_meteo(ruta_a_carpeta, archivo)
                df_pkl = pd.concat([df_pkl, df_pkl_crudo_realt], ignore_index=True)
    
df_pkl = df_pkl.reset_index(drop=True)  # Reiniciar el índice del DataFrame combinado
df_pkl = df_pkl[["tspan","Pa","Ta","HR"]]
dict_de_boyas[nombre] = df_pkl # Guardo el dataframe combinado dentro del diccionario de boyas.

In [11]:
# Crear el diccionario de boyas con los dataframes cargados
df = crear_dataframe_de_boyas(dict_de_boyas)

Index(['tspan', 'Pa_BMT2-01-T20_pkl', 'Ta_BMT2-01-T20_pkl',
       'Humedad relativa (sonico)_BMT2-01-T20_pkl',
       'Punto de Rocio_BMT2-01-T20_pkl', 'Rap2_BMT2-01-T20_pkl',
       'Dir2_BMT2-01-T20_pkl', 'u (mecanico)_BMT2-01-T20_pkl',
       'v (mecanico)_BMT2-01-T20_pkl', 'R5s2_BMT2-01-T20_pkl',
       'R1s2_BMT2-01-T20_pkl', 'Rap1_BMT2-01-T20_pkl', 'Dir1_BMT2-01-T20_pkl',
       'u (sonico)_BMT2-01-T20_pkl', 'v (sonico)_BMT2-01-T20_pkl',
       'R5s1_BMT2-01-T20_pkl', 'R1s1_BMT2-01-T20_pkl',
       'Temperatura (sonico)_BMT2-01-T20_pkl',
       'Presion (sonico)_BMT2-01-T20_pkl', 'HR_BMT2-01-T20_pkl',
       'Lluvia acumulada_BMT2-01-T20_pkl',
       'Duracion de la lluvia_BMT2-01-T20_pkl',
       'Intensidad de la lluvia_BMT2-01-T20_pkl', 'Pa_BMT2-01-T20_NC',
       'Ta_BMT2-01-T20_NC', 'HR_BMT2-01-T20_NC', 'Rap1_BMT2-01-T20_NC',
       'Dir1_BMT2-01-T20_NC', 'R5s1_BMT2-01-T20_NC', 'R1s1_BMT2-01-T20_NC',
       'R5s2_BMT2-01-T20_NC', 'R1s2_BMT2-01-T20_NC', 'Rap2_BMT2-01-T20_NC'

In [12]:
app = InteractivePlotterTk(df)
# Iniciar el bucle de la aplicación
app.root.mainloop()

# NOTAS PENDIENTES:
# - Arreglar que al eliminar puntos para que no tenga límite el historial de deshacer
# - Arreglar para que solo se grafiquen las columnas elegidas por el usuario
# - Agregar una función que grafique las variables de Rap y Dir --> sería algo como que df.iloc[idx,col][prof]
# - Arreglar el caso de cuando el usuario elimina puntos de Rap o Dir colocar U y V como nans --> df.iloc[idx,col][prof] = np.nan
# - Agregar la funcionalidad de guardar el DataFrame corregido en un archivo .csv o .xlsx, pkl
# - Agregar la funcionalidad de devolver puntos eliminados (no deshacer la última acción, sino devolver puntos específicos)